# Paso 3 (v3) – Reducción de columnas y homologación de diagnósticos

**Proyecto:** seguimiento de residentes del Puericultorio Pérez Araníbar.

Este cuaderno parte de `Dataset_unificado_2.xlsx` y aplica seis tratamientos adicionales:

| # | Tratamiento | Criterio |
|---|---|---|
| 1 | **Eliminar columnas de auditoría** | `origen_registro`, `estado_union` y `nutricion_excluida_motivo` solo documentan cómo se armó el consolidado. |
| 2 | **Excluir residentes con cronología incompatible** | Se quitan todos los registros de los NNA cuya fecha de ingreso es anterior a su fecha de nacimiento. |
| 3 | **Unificar las categorías de los diagnósticos** | Una sola escritura por categoría antes de comparar curva y tabla. |
| 4 | **Conservar el diagnóstico de mayor riesgo** | De cada par curva/tabla (P/E, T/E, P/T, IMC) se queda el más alejado de lo normal. Resultado: `dx_pe`, `dx_te`, `dx_pt`, `dx_imc`. |
| 5 | **Estandarizar la nacionalidad** | `PERUANA` y `PERUANO` pasan a `PERUANO`. |
| 6 | **Homologar situación legal, UPE e institución educativa** | Una sola escritura por categoría, para que el Paso 4 lea las columnas ya estandarizadas. |

El resultado se guarda en `Dataset_unificado_3.xlsx`. El archivo de entrada no se modifica.

# 1. Lectura del consolidado 2

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Raíz del proyecto: este cuaderno se guarda en la carpeta Notebooks
BASE = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
RUTA_ENTRADA = BASE / 'Resultados' / 'Consolidado' / 'Dataset_unificado_2.xlsx'
RUTA_SALIDA = BASE / 'Resultados' / 'Consolidado' / 'Dataset_unificado_3.xlsx'
RUTA_EVID = BASE / 'Resultados' / 'Paso3_evidencias'
RUTA_EVID.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(RUTA_ENTRADA)
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head(3)

Filas: 2,535 | Columnas: 50


,id_registro,id_persona,periodo,origen_registro,estado_union,nutricion_excluida_motivo,sexo,fecha_nacimiento,nom_Departamento,nom_Provincia,nom_Distrito,nom_Nacionalidad,nom_Fecha de ingreso,nom_Observaciones y/o comentarios de la forma de ingreso de la NNA,nom_Tipo de documento,nom_Fecha de la Resolucion Administrativa,nom_Situación Legal de la (NNA),nom_Fecha del Oficio de Ingreso,nom_Nombre de UPE /Juzgado,nom_FECHA ELABORACIÓN Plan de Trabajo individual - PTI,nom_VIGENCIA DEL PTI,nom_ESTADO DEL PTI,nom_Instancia que elabora el PTI,nom_Grado de Instrucción,nom_Último nivel educativo logrado,nom_Tipo de Institución Educativa,nom_Acceso a capacitación técnico productiva,nom_La NNA actualmente estudia,nom_Estado de Salud Física Actual,nom_Presenta Discapacidad (Indicar tipo de discapacidad),nom_Nivel Nutricional,nom_Soporte Psico-emocional,nom_Atenciones de salud psicológica,nom_Atención en Salud mental especializada,nom_Seguro de Salud (NNA),nut_id_evaluacion,nut_grupo_hogar,nut_casa,nut_peso_kg,nut_talla_cm,nut_dx_pe_curva,nut_dx_te_curva,nut_dx_pt_curva,nut_dx_imc_curva,nut_dx_pe_tabla,nut_dx_te_tabla,nut_dx_pt_tabla,nut_dx_imc_tabla,nut_imc_completado,nut_edad_meses_usada_para_tablas_ins
0,NOM_00002,PPA_FN_0001,2025-01,solo_nominal,sin_nutricion_en_periodo,NaN,F,2022-03-26,LIMA,LIMA,LIMA,PERUANA,2022-09-06,Presunta desprotección familiar,Resolución Admi.,2022-09-06,Desprotección Familiar Provisional,2022-09-06,UPE LIMA NORTE CALLAO,2024-10-16,mientras dure su proceso legalpara su adoptabi...,Actual,Upe Lima Norte Callao,Inicial,Inicial,Educación Básica Regular -EBR,No aplica,Si,Saludable,No tiene,Sobrepeso,Acompañamiento/Estimulación,NO,No aplica,SIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NOM_00003,PPA_FN_0002,2025-01,solo_nominal,sin_nutricion_en_periodo,NaN,M,2021-11-30,LIMA,LIMA,CALLAO,PERUANO,2021-12-13,Presunta desprotección familiar,Resolución Admi.,2021-12-13,Desprotección Familiar,2021-12-13,UPE LIMA NORTE CALLAO,2023-01-19,2023-12-19,Vencido,Upe Lima Norte Callao,Ninguno,Ninguno,Ninguno,No aplica,No,Saludable,No tiene,Saludable,Acompañamiento/Estimulación,NO,No aplica,SIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NOM_00004,PPA_FN_0003,2025-01,solo_nominal,sin_nutricion_en_periodo,NaN,M,2022-03-28,LIMA,LIMA,CALLAO,PERUANO,2022-04-04,Presunta desprotección familiar,Resolución Admi.,2022-04-04,Desprotección Familiar,2022-04-04,UPE LIMA NORTE CALLAO,2024-01-10,2024-11-10,Vencido,Upe Lima Norte Callao,Ninguno,Ninguno,Ninguno,No aplica,No,Saludable,No tiene,Riesgo de baja talla,Acompañamiento/Estimulación,NO,No aplica,SIS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Comentario 1:** el consolidado 2 tiene 2 535 registros (un NNA por mes) y 50 columnas. Cada fila combina la ficha nominal (`nom_*`) con la evaluación nutricional del mismo mes (`nut_*`), cuando existe. Como la nutrición no se evaluó todos los meses, la mayoría de las columnas `nut_*` están vacías por diseño y no por error.

# 2. Eliminación de columnas de auditoría

In [ ]:
COL_AUDITORIA = ['origen_registro', 'estado_union', 'nutricion_excluida_motivo']

print(df[COL_AUDITORIA].apply(lambda s: s.value_counts(dropna=False).to_dict()).to_string())

n_antes = df.shape[1]
df = df.drop(columns=COL_AUDITORIA)
print(f'\nColumnas: {n_antes} -> {df.shape[1]}')

origen_registro              {'solo_nominal': 2180, 'nominal_y_nutricion': ...
estado_union                 {'sin_nutricion_en_periodo': 2180, 'coincidenc...
nutricion_excluida_motivo                         {nan: 2512, 'PROVINCIA': 23}

Columnas: 50 -> 47


**Comentario 2:** las tres columnas solo registran el proceso de construcción del consolidado: si la fila viene solo de la ficha nominal o también de nutrición, cómo se unieron y por qué se excluyó una evaluación (`PROVINCIA`). No describen al residente, y dejarlas haría que el clustering agrupe por cómo se armó el archivo y no por las características de los NNA. Esa trazabilidad se conserva en `Dataset_unificado_2.xlsx` y en las evidencias del Paso 3, por lo que se eliminan sin perder información. Quedan 47 columnas.

# 3. Exclusión de residentes con cronología incompatible

In [ ]:
f_nac, f_ing = 'fecha_nacimiento', 'nom_Fecha de ingreso'

incompat = df[f_ing] < df[f_nac]
ids_cron = sorted(df.loc[incompat, 'id_persona'].unique())

detalle = df[df['id_persona'].isin(ids_cron)].groupby('id_persona').agg(
    registros=('periodo', 'size'),
    fecha_nacimiento=(f_nac, 'first'),
    fecha_ingreso=(f_ing, 'first'),
)
detalle['registros_incompatibles'] = df[incompat].groupby('id_persona').size()
detalle['dias_ingreso_antes_nacimiento'] = (detalle['fecha_nacimiento'] - detalle['fecha_ingreso']).dt.days
print(f'Registros con ingreso < nacimiento: {incompat.sum()} | Residentes afectados: {len(ids_cron)}')
detalle

Registros con ingreso < nacimiento: 76 | Residentes afectados: 4


,registros,fecha_nacimiento,fecha_ingreso,registros_incompatibles,dias_ingreso_antes_nacimiento
id_persona,,,,,
PPA_FN_0004,19,2023-09-01,2023-01-16,19,228
PPA_FN_0033,19,2014-04-12,2014-03-06,19,37
PPA_FN_0034,19,2014-04-12,2014-03-06,19,37
PPA_FN_0073,19,2014-05-17,2014-01-08,19,129


**Comentario 3:** hay 4 residentes (PPA_FN_0004, 0033, 0034 y 0073) cuya fecha de ingreso es anterior a su fecha de nacimiento, lo cual es imposible: una persona no puede ingresar a la institución antes de nacer. En los 4 casos ocurre en los 19 meses de la ficha (76 registros), por lo que no es un error de un mes aislado sino de un dato que se arrastra en todo su historial. Los desfases van de unas semanas a unos meses, lo que apunta a un error de digitación en alguna de las dos fechas, pero no hay forma de saber cuál es la correcta. Como ambas fechas determinan la edad y el tiempo desde el ingreso, que son variables centrales del análisis, no es seguro conservarlas ni corregirlas sin consultar a la institución. Por eso se excluyen todos los registros de estos residentes y se guardan como evidencia para que el Puericultorio pueda revisarlos y corregirlos.

In [ ]:
excluidos_cron = df[df['id_persona'].isin(ids_cron)].copy()
df = df[~df['id_persona'].isin(ids_cron)].reset_index(drop=True)

assert not (df[f_ing] < df[f_nac]).any()
print(f'Registros excluidos: {len(excluidos_cron)} | Registros restantes: {len(df):,} | Residentes restantes: {df["id_persona"].nunique()}')

Registros excluidos: 76 | Registros restantes: 2,459 | Residentes restantes: 145


**Comentario 4:** tras la exclusión ya no queda ningún registro con ingreso anterior al nacimiento. Se retiran 76 registros y el consolidado queda con 2 459, sin alterar el resto de la información. Las filas retiradas se guardan completas en la carpeta de evidencias.

# 4. Homologación de las categorías de diagnóstico

In [ ]:
BASES = ['pe', 'te', 'pt', 'imc']
COL_DX = [f'nut_dx_{b}_{t}' for b in BASES for t in ('curva', 'tabla')]

for c in COL_DX:
    print(c, df[c].value_counts().to_dict())

nut_dx_pe_curva {'NORMAL': 74, 'RIESGO SOBREPESO': 19, 'SOBREPESO': 13, 'RIESGO BAJO PESO': 9, 'OBESIDAD': 5, 'BAJO PESO': 4, 'BAJO PESO SEVERO': 1}
nut_dx_pe_tabla {'NORMAL': 51, 'SOBREPESO': 5, 'RIESGO BAJO PESO': 2, 'DESNUTRIDO': 1, 'OBESIDAD': 1}
nut_dx_te_curva {'NORMAL': 175, 'RIESGO TALLA BAJA': 75, 'TALLA BAJA': 51, 'TALLA BAJA SEVERA': 26, 'ALTO': 14}
nut_dx_te_tabla {'NORMAL': 263, 'RIESGO TALLA BAJA': 50, 'TALLA BAJA': 19, 'TALLA BAJA SEVERA': 6, 'ALTO': 4}
nut_dx_pt_curva {'NORMAL': 21, 'RIESGO SOBREPESO': 13, 'SOBREPESO': 8, 'OBESIDAD': 6, 'BAJO PESO': 1, 'RIESGO OBESIDAD': 1}
nut_dx_pt_tabla {'NORMAL': 47, 'SOBREPESO': 6, 'RIESGO SOBREPESO': 5, 'OBESIDAD': 4}
nut_dx_imc_curva {'NORMAL': 154, 'RIESGO SOBREPESO': 83, 'SOBREPESO': 54, 'OBESIDAD': 22, 'RIESGO BAJO PESO': 9, 'BAJO PESO': 3, 'RIESGO DE DELGADEZ O RIESGO DE BAJO PESO': 1}
nut_dx_imc_tabla {'NORMAL': 163, 'RIESGO SOBREPESO': 73, 'SOBREPESO': 36, 'OBESIDAD': 10, 'OBESIDAD III': 2, 'RIESGO DE DELGADEZ O RIESGO DE B

**Comentario 5:** antes de comparar curva con tabla hay que verificar que ambas usen las mismas etiquetas. Hay dos casos en los que un mismo estado se escribe distinto: `DESNUTRIDO` (tabla INS de peso para la edad) equivale a `BAJO PESO`, y `RIESGO DE DELGADEZ O RIESGO DE BAJO PESO` (IMC) equivale a `RIESGO BAJO PESO`, que es la forma que se usa en el resto de columnas. Si no se unificaran, la comparación los trataría como diagnósticos distintos y generaría diferencias falsas.

In [ ]:
UNIFICAR = {
    'DESNUTRIDO': 'BAJO PESO',
    'RIESGO DE DELGADEZ O RIESGO DE BAJO PESO': 'RIESGO BAJO PESO',
}
for c in COL_DX:
    df[c] = df[c].replace(UNIFICAR)

pd.DataFrame({c: df[c].value_counts() for c in COL_DX}).fillna(0).astype(int)

,nut_dx_pe_curva,nut_dx_pe_tabla,nut_dx_te_curva,nut_dx_te_tabla,nut_dx_pt_curva,nut_dx_pt_tabla,nut_dx_imc_curva,nut_dx_imc_tabla
ALTO,0,0,14,4,0,0,0,0
BAJO PESO,4,1,0,0,1,0,3,0
BAJO PESO SEVERO,1,0,0,0,0,0,0,0
NORMAL,74,51,175,263,21,47,154,163
OBESIDAD,5,1,0,0,6,4,22,10
OBESIDAD III,0,0,0,0,0,0,0,2
RIESGO BAJO PESO,9,2,0,0,0,0,10,1
RIESGO OBESIDAD,0,0,0,0,1,0,0,0
RIESGO SOBREPESO,19,0,0,0,13,5,83,73
RIESGO TALLA BAJA,0,0,75,50,0,0,0,0


# 5. Conservar el diagnóstico de mayor riesgo

## 5.1 Escala de riesgo

Para decidir cuál de los dos diagnósticos es "más riesgoso" se asigna un nivel a cada categoría según qué tan lejos está de lo normal, sin importar si es por déficit o por exceso:

| Nivel | Categorías |
|---|---|
| 0 | NORMAL |
| 0.5 | ALTO (talla alta; desvío leve que no implica un problema nutricional) |
| 1 | RIESGO BAJO PESO, RIESGO SOBREPESO, RIESGO TALLA BAJA |
| 2 | BAJO PESO, SOBREPESO, TALLA BAJA, RIESGO OBESIDAD |
| 3 | BAJO PESO SEVERO, TALLA BAJA SEVERA, OBESIDAD |
| 4 | OBESIDAD III |

**Comentario 6:** la escala sigue la gravedad de cada categoría dentro de su indicador. `ALTO` se ubica por debajo de cualquier categoría de riesgo, para que frente a `RIESGO TALLA BAJA` prevalezca el segundo. `RIESGO OBESIDAD` (un solo caso en P/T) se ubica con el sobrepeso por ser un estado ya definido y no una simple alerta. Si en una comparación ambos diagnósticos tienen el mismo nivel pero distinta etiqueta, se conserva el de la curva; esta regla se programa, pero se verifica más abajo que no llegó a usarse.

In [ ]:
NIVEL = {
    'NORMAL': 0, 'ALTO': 0.5,
    'RIESGO BAJO PESO': 1, 'RIESGO SOBREPESO': 1, 'RIESGO TALLA BAJA': 1,
    'BAJO PESO': 2, 'SOBREPESO': 2, 'TALLA BAJA': 2, 'RIESGO OBESIDAD': 2,
    'BAJO PESO SEVERO': 3, 'TALLA BAJA SEVERA': 3, 'OBESIDAD': 3,
    'OBESIDAD III': 4,
}

sin_nivel = set()
for c in COL_DX:
    sin_nivel |= set(df[c].dropna()) - set(NIVEL)
print('Categorías sin nivel asignado:', sin_nivel or 'ninguna')

Categorías sin nivel asignado: ninguna


## 5.2 Comparación curva vs. tabla

In [ ]:
resumen = []
conflictos = []

for b in BASES:
    curva, tabla = df[f'nut_dx_{b}_curva'], df[f'nut_dx_{b}_tabla']
    n_c, n_t = curva.map(NIVEL), tabla.map(NIVEL)

    ambos = curva.notna() & tabla.notna()
    difieren = ambos & (curva != tabla)
    tabla_mayor = ambos & (n_t > n_c)
    curva_mayor = ambos & (n_c > n_t)
    empate = difieren & (n_c == n_t)
    solo_curva = curva.notna() & tabla.isna()
    solo_tabla = curva.isna() & tabla.notna()

    # Se usa la tabla cuando tiene mayor nivel o cuando es la única disponible
    usar_tabla = tabla_mayor | solo_tabla
    final = pd.Series(np.where(usar_tabla, tabla, curva), index=df.index)
    final[curva.isna() & tabla.isna()] = np.nan
    df[f'dx_{b}'] = final

    resumen.append({
        'indicador': b, 'con_ambos': ambos.sum(), 'coinciden': (ambos & ~difieren).sum(),
        'difieren': difieren.sum(), 'gana_curva': curva_mayor.sum(), 'gana_tabla': tabla_mayor.sum(),
        'empates_por_nivel': empate.sum(), 'solo_curva': solo_curva.sum(), 'solo_tabla': solo_tabla.sum(),
        'dx_final_no_nulos': final.notna().sum(),
    })

    c = df.loc[difieren, ['id_registro', 'id_persona', 'periodo']].copy()
    c['indicador'] = b
    c['curva'], c['tabla'] = curva[difieren], tabla[difieren]
    c['conservado'] = final[difieren]
    conflictos.append(c)

resumen = pd.DataFrame(resumen).set_index('indicador')
conflictos = pd.concat(conflictos, ignore_index=True)
resumen

,con_ambos,coinciden,difieren,gana_curva,gana_tabla,empates_por_nivel,solo_curva,solo_tabla,dx_final_no_nulos
indicador,,,,,,,,,
pe,59,43,16,16,0,0,66,1,126
te,341,213,128,123,5,0,0,1,342
pt,49,26,23,22,1,0,1,13,63
imc,283,208,75,62,13,0,43,2,328


**Comentario 7:** en los cuatro indicadores hay evaluaciones donde la curva (OMS) y la tabla (INS) dan un diagnóstico distinto: 16 en P/E, 128 en T/E, 23 en P/T y 75 en IMC. Esto es esperable, porque para los mayores de 5 años la tabla del INS se basa en la referencia CDC 2000 y no en la OMS (ver Paso 2). Al quedarse con el de mayor riesgo, la curva prevalece en la mayoría de los casos y la tabla en una minoría (5 en T/E, 1 en P/T y 13 en IMC). No hubo empates entre diagnósticos distintos, por lo que la regla de desempate no se aplicó. Además, la tabla es la única fuente en 1 caso de P/E, 1 de T/E, 13 de P/T y 2 de IMC, donde se toma ese valor porque no hay nada con qué compararlo. El criterio es conservador, pues evita que un riesgo detectado por una de las dos referencias se pierda al elegir solo una.

In [ ]:
print('Muestra de diferencias (curva vs tabla):')
conflictos.sort_values(['indicador', 'id_persona', 'periodo']).head(15)

Muestra de diferencias (curva vs tabla):


,id_registro,id_persona,periodo,indicador,curva,tabla,conservado
198,NOM_02409,PPA_FN_0013,2026-06,imc,RIESGO SOBREPESO,NORMAL,RIESGO SOBREPESO
199,NOM_02411,PPA_FN_0015,2026-06,imc,SOBREPESO,RIESGO SOBREPESO,SOBREPESO
200,NOM_02412,PPA_FN_0016,2026-06,imc,RIESGO SOBREPESO,NORMAL,RIESGO SOBREPESO
201,NOM_02413,PPA_FN_0017,2026-06,imc,SOBREPESO,RIESGO SOBREPESO,SOBREPESO
188,NOM_02145,PPA_FN_0018,2026-04,imc,RIESGO BAJO PESO,NORMAL,RIESGO BAJO PESO
202,NOM_02414,PPA_FN_0018,2026-06,imc,BAJO PESO,NORMAL,BAJO PESO
167,NOM_01740,PPA_FN_0022,2026-01,imc,RIESGO BAJO PESO,NORMAL,RIESGO BAJO PESO
203,NOM_02418,PPA_FN_0022,2026-06,imc,SOBREPESO,NORMAL,SOBREPESO
204,NOM_02419,PPA_FN_0023,2026-06,imc,SOBREPESO,RIESGO SOBREPESO,SOBREPESO
189,NOM_02152,PPA_FN_0024,2026-04,imc,SOBREPESO,RIESGO SOBREPESO,SOBREPESO


In [ ]:
# Verificación: el diagnóstico final tiene siempre el mayor nivel de riesgo entre curva y tabla
for b in BASES:
    nf = df[f'dx_{b}'].map(NIVEL)
    nmax = pd.concat([df[f'nut_dx_{b}_curva'].map(NIVEL), df[f'nut_dx_{b}_tabla'].map(NIVEL)], axis=1).max(axis=1)
    assert (nf.fillna(-1) == nmax.fillna(-1)).all(), b
print('OK: dx_* coincide en todos los casos con el mayor nivel de riesgo entre curva y tabla.')

# Se retiran las 8 columnas de origen y se deja el diagnóstico final en su lugar
pos = df.columns.get_loc('nut_dx_pe_curva')
df = df.drop(columns=COL_DX)
nuevas = ['dx_pe', 'dx_te', 'dx_pt', 'dx_imc']
cols = [c for c in df.columns if c not in nuevas]
cols[pos:pos] = nuevas
df = df[cols]
print(f'Columnas: {df.shape[1]}')
pd.DataFrame({f'dx_{b}': df[f'dx_{b}'].value_counts() for b in BASES}).fillna(0).astype(int)

OK: dx_* coincide en todos los casos con el mayor nivel de riesgo entre curva y tabla.
Columnas: 43


,dx_pe,dx_te,dx_pt,dx_imc
ALTO,0,15,0,0
BAJO PESO,4,0,1,3
BAJO PESO SEVERO,1,0,0,0
NORMAL,75,171,32,147
OBESIDAD,5,0,7,23
OBESIDAD III,0,0,0,2
RIESGO BAJO PESO,9,0,0,10
RIESGO OBESIDAD,0,0,1,0
RIESGO SOBREPESO,19,0,14,85
RIESGO TALLA BAJA,0,77,0,0


**Comentario 8:** las 8 columnas de diagnóstico se reducen a 4 (`dx_pe`, `dx_te`, `dx_pt`, `dx_imc`), sin indicar de qué referencia proviene cada valor, como se pidió. Se verificó que en todos los registros el valor final coincide con el mayor nivel de riesgo entre curva y tabla. El detalle de cada diferencia (curva, tabla y valor conservado) queda guardado como evidencia, de modo que la decisión sigue siendo revisable. Las columnas se reemplazan en la misma posición para mantener el orden de las variables nutricionales. Los diagnósticos siguen vacíos en los meses sin evaluación, lo que se tratará al construir la matriz analítica por residente.

# 6. Estandarización de la nacionalidad

In [ ]:
col_nac = 'nom_Nacionalidad'
print('Antes:', df[col_nac].value_counts(dropna=False).to_dict())

df[col_nac] = df[col_nac].replace({'PERUANA': 'PERUANO'})

print('Después:', df[col_nac].value_counts(dropna=False).to_dict())

Antes: {'PERUANA': 1459, 'PERUANO': 963, nan: 37}
Después: {'PERUANO': 2422, nan: 37}


**Comentario 9:** `PERUANA` (1 459 registros) y `PERUANO` (963) expresan la misma nacionalidad, escrita según el género gramatical y no según un dato distinto. Dejarlas separadas haría que el clustering las trate como dos categorías, y la diferencia entre ambas no aportaría información sobre el residente sino sobre la escritura del registro. Tras unificarlas queda una sola categoría con 2 422 registros y 37 vacíos, que se conservan sin imputar porque esa decisión corresponde a la etapa de transformación.

# 7. Homologación de situación legal, UPE e institución educativa

In [ ]:
import re
import unicodedata

def _norm(x):
    if pd.isna(x):
        return np.nan
    t = unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'\s+', ' ', t).strip()

def hom_situacion(x):
    t = _norm(x)
    if pd.isna(t): return np.nan
    if 'provisional' in t: return 'Provisional'
    if 'definitiva' in t: return 'Definitiva'
    if 'urgente' in t: return 'Urgencia'
    return 'No estandarizado'

def hom_upe(x):
    t = _norm(x)
    if pd.isna(t): return np.nan
    if 'norte' in t: return 'Lima Norte Callao'
    if 'este' in t: return 'Lima Este'
    if 'sur' in t: return 'Lima Sur'
    if 'centro' in t: return 'Lima Centro'
    if t == 'upe lima': return 'Lima'
    return 'Otra instancia'

def hom_institucion(x):
    t = _norm(x)
    if pd.isna(t): return np.nan
    if t == 'ninguno': return 'Ninguna'
    if 'alternativa' in t: return 'Otra modalidad'
    if 'regular' in t or t.endswith('ebr'): return 'Regular (EBR)'
    return 'Otra modalidad'

HOMOLOGAR = {
    'nom_Situación Legal de la (NNA)': hom_situacion,
    'nom_Nombre de UPE /Juzgado': hom_upe,
    'nom_Tipo de Institución Educativa': hom_institucion,
}

equivalencias = []
for col, fn in HOMOLOGAR.items():
    original = df[col].copy()
    df[col] = original.map(fn)
    t = pd.DataFrame({'original': original, 'homologado': df[col]}).value_counts(dropna=False).rename('registros').reset_index()
    t.insert(0, 'columna', col)
    equivalencias.append(t)
    print(col)
    print(df[col].value_counts(dropna=False).to_dict(), '\n')

equivalencias = pd.concat(equivalencias, ignore_index=True)
equivalencias

nom_Situación Legal de la (NNA)
{'Provisional': 1984, 'Definitiva': 211, 'Urgencia': 173, 'No estandarizado': 54, nan: 37} 

nom_Nombre de UPE /Juzgado
{'Lima Norte Callao': 785, 'Lima Este': 734, 'Lima': 646, 'Lima Sur': 186, 'Otra instancia': 52, nan: 37, 'Lima Centro': 19} 

nom_Tipo de Institución Educativa
{'Regular (EBR)': 2135, 'Ninguna': 175, 'Otra modalidad': 112, nan: 37} 



,columna,original,homologado,registros
0,nom_Situación Legal de la (NNA),Desprotección Familiar Provisional,Provisional,1984
1,nom_Situación Legal de la (NNA),Desprotección Familiar Definitiva,Definitiva,211
2,nom_Situación Legal de la (NNA),Acogimiento residencial en calidad de urgente,Urgencia,173
3,nom_Situación Legal de la (NNA),NaN,NaN,37
4,nom_Situación Legal de la (NNA),Desprotección Familiar,No estandarizado,24
5,nom_Situación Legal de la (NNA),DECIMO PRIMER JUZGADO DE FAMILIA DE LIMA,No estandarizado,19
6,nom_Situación Legal de la (NNA),Auto de abandono,No estandarizado,11
7,nom_Nombre de UPE /Juzgado,UPE LIMA NORTE CALLAO,Lima Norte Callao,785
8,nom_Nombre de UPE /Juzgado,UPE LIMA ESTE,Lima Este,734
9,nom_Nombre de UPE /Juzgado,UPE LIMA,Lima,646


**Comentario 10:** estas tres columnas tenían la misma categoría escrita de varias formas o mezclaban valores que no son categorías. Se homologan en el mismo lugar (sin crear columnas nuevas) para que el Paso 4 las reciba ya estandarizadas:
- **Situación legal:** se reduce a `Provisional`, `Definitiva` y `Urgencia`. El genérico "Desprotección Familiar" (24 registros), el nombre de un juzgado (19) y "Auto de abandono" (11) no corresponden a una situación legal estandarizada, por lo que pasan a `No estandarizado` en lugar de forzarlos a una categoría; esto coincide con los valores sin estandarizar que se señalaron en el Paso 2.
- **UPE / juzgado:** se unen las variantes de escritura (`UPE-LIMA-SUR` con `UPE LIMA SUR`) y las instancias que no son una UPE (juzgado, UIT, DGA) pasan a `Otra instancia`.
- **Tipo de institución educativa:** se agrupa en `Regular (EBR)`, `Otra modalidad` (CEBA, CEBE, técnico, taller) y `Ninguna`. La "Educación Básica Alternativa" que venía rotulada como EBR se clasifica como `Otra modalidad`.

Las categorías originales y su equivalencia quedan en la tabla anterior y se guardan como evidencia. Los vacíos (los 37 registros sin ficha) se mantienen.

# 8. Verificación y guardado

In [ ]:
print(f'Entrada : {RUTA_ENTRADA.name}  -> 2 535 filas x 50 columnas')
print(f'Salida   : {RUTA_SALIDA.name}  -> {df.shape[0]:,} filas x {df.shape[1]} columnas ({len(excluidos_cron)} filas excluidas por cronología)')
assert df.shape[0] == 2535 - len(excluidos_cron) and df['id_registro'].is_unique

print('Columnas retiradas:', COL_AUDITORIA + COL_DX)
print('Columnas nuevas   :', ['dx_pe', 'dx_te', 'dx_pt', 'dx_imc'])

df.to_excel(RUTA_SALIDA, index=False)
resumen.to_excel(RUTA_EVID / '11_resumen_comparacion_curva_tabla.xlsx')
conflictos.to_excel(RUTA_EVID / '12_diferencias_curva_vs_tabla.xlsx', index=False)
excluidos_cron.to_excel(RUTA_EVID / '13_cronologia_incompatible_excluidos.xlsx', index=False)
equivalencias.to_excel(RUTA_EVID / '14_homologacion_legal_upe_institucion.xlsx', index=False)
print('Guardado.')

Entrada : Dataset_unificado_2.xlsx  -> 2 535 filas x 50 columnas
Salida   : Dataset_unificado_3.xlsx  -> 2,459 filas x 43 columnas (76 filas excluidas por cronología)
Columnas retiradas: ['origen_registro', 'estado_union', 'nutricion_excluida_motivo', 'nut_dx_pe_curva', 'nut_dx_pe_tabla', 'nut_dx_te_curva', 'nut_dx_te_tabla', 'nut_dx_pt_curva', 'nut_dx_pt_tabla', 'nut_dx_imc_curva', 'nut_dx_imc_tabla']
Columnas nuevas   : ['dx_pe', 'dx_te', 'dx_pt', 'dx_imc']


Guardado.


**Comentario 11:** se pasa de 2 535 a 2 459 registros: la diferencia son exactamente las 76 filas excluidas por cronología incompatible, y el identificador de registro sigue siendo único, es decir, no se perdieron ni duplicaron otras filas. El consolidado pasa de 50 a 43 columnas: se quitan 3 de auditoría y 8 de diagnóstico y se agregan 4. Se guardan cinco archivos: `Dataset_unificado_3.xlsx` (dataset final de este paso) y, como evidencias, `11_resumen_comparacion_curva_tabla.xlsx`, `12_diferencias_curva_vs_tabla.xlsx`, `13_cronologia_incompatible_excluidos.xlsx` y `14_homologacion_legal_upe_institucion.xlsx`.